In [59]:
# Move dummy data from dummy_dataset to hdfs using hdfs connect
# Then we implement analyze this data suing sparksql

# - Ex1: Count total item
#- Ex2: Show hot items
#-  Ex3: Show only item be purchased only once
#- Ex3: 

In [116]:
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 37.1 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [118]:
from hdfs import InsecureClient
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import explode, array, col, concat_ws, when, lit, split
from pyspark.ml.fpm import FPGrowth
from datetime import datetime

In [95]:
HDFS_WEB_BROWSER = "http://dists-hdfs-namenode:9870"
HDFS_NAMENODE = "hdfs://dists-hdfs-namenode:8020"

SPARK_APP_NAME = "team-de-spark-practice-25-04-v1.0.5"
SPARK_MASTER = "spark://ingest-spark-master:7077"

In [62]:
client = InsecureClient(HDFS_WEB_BROWSER, user="root")
client.status("/")

{'accessTime': 0,
 'blockSize': 0,
 'childrenNum': 8,
 'fileId': 16385,
 'group': 'supergroup',
 'length': 0,
 'modificationTime': 1777107636116,
 'owner': 'root',
 'pathSuffix': '',
 'permission': '755',
 'replication': 0,
 'snapshotEnabled': True,
 'storagePolicy': 0,
 'type': 'DIRECTORY'}

In [96]:
spark = (SparkSession.builder.appName(SPARK_APP_NAME).master(SPARK_MASTER).getOrCreate())
spark

In [97]:
# Move dummy data to hdfs
DUMMY_DESTITAION_DIR = "/assiation-rule-mining"

DUMMY_DATASET_DIR = "./dummy_dataset/shop.csv"
DUMMY_DATASET_CUSTOMER_DIR = "./dummy_dataset/customer.csv"
DUMMY_DATASET_STUDENT_SCORE_DIR = "./dummy_dataset/student_score.csv"

DUMMY_DESTINATION_FILE = DUMMY_DESTITAION_DIR + "/shop.csv"
DUMMY_DESTINATION_FILE_CUSTOMER = DUMMY_DESTITAION_DIR + "/customer.csv"
DUMMY_DESTINATION_FILE_STUDENT_SCORE = DUMMY_DESTITAION_DIR + "/student_score.csv"

DUMMY_DESTINATION_FILE_FULL_PATH = HDFS_NAMENODE + DUMMY_DESTINATION_FILE
DUMMY_DESTINATION_FILE_CUSTOMER_FULL_PATH = HDFS_NAMENODE + DUMMY_DESTINATION_FILE_CUSTOMER
DUMMY_DESTINATION_FILE_STUDENT_SCORE_FULL_PATH = HDFS_NAMENODE + DUMMY_DESTINATION_FILE_STUDENT_SCORE

client.makedirs(DUMMY_DESTITAION_DIR)

client.upload(DUMMY_DESTITAION_DIR, DUMMY_DATASET_DIR, overwrite=True)
client.upload(DUMMY_DESTINATION_FILE_CUSTOMER, DUMMY_DATASET_CUSTOMER_DIR, overwrite=True)
client.upload(DUMMY_DESTINATION_FILE_STUDENT_SCORE, DUMMY_DATASET_STUDENT_SCORE_DIR, overwrite=True)

client.list(DUMMY_DESTITAION_DIR)

['customer.csv', 'reports', 'shop.csv', 'student_score.csv']

In [25]:
print(DUMMY_DESTINATION_FILE_FULL_PATH)

schema = StructType([
    StructField("item1", StringType(), True),
    StructField("item2", StringType(), True),
    StructField("item3", StringType(), True)
])

df = spark.read.csv(DUMMY_DESTINATION_FILE_FULL_PATH, header=False, schema=schema)
df.show()

hdfs://dists-hdfs-namenode:8020/assiation-rule-mining/shop.csv


[Stage 0:>                                                          (0 + 1) / 1]

+------+-------+------+
| item1|  item2| item3|
+------+-------+------+
|  milk|  bread|  eggs|
| bread| butter|  NULL|
|  milk|  bread|butter|
|cheese|   eggs|  NULL|
| bread|   NULL|  NULL|
|butter| cheese|  NULL|
|  eggs|   NULL|  NULL|
|  milk| cheese|  eggs|
| bread| butter|  eggs|
|  milk|chicken|  NULL|
+------+-------+------+



In [26]:
# Count each items.
df_count = df.select(explode(array("item1", "item2", "item3")).alias("item"))
df_count = df_count.filter(col("item").isNotNull())
df_count = df_count.groupBy("item").count().orderBy("count", ascending=False)
df_count.show()

+-------+-----+
|   item|count|
+-------+-----+
|   eggs|    5|
|  bread|    5|
|   milk|    4|
| butter|    4|
| cheese|    3|
|chicken|    1|
+-------+-----+



In [29]:
# Contiue  Lấy ra top 10 mặt hàng chỉ được đặt duy nhất 1 lần
df_purchase_only_once = df_count.filter(col("count") == 1)
df_purchase_only_once.show()

+-------+-----+
|   item|count|
+-------+-----+
|chicken|    1|
+-------+-----+



In [68]:
# Excercise 2
# inferSchema is an option in Spark (especially when reading CSV/JSON) that tells 
# Spark to automatically detect the data types of each column instead of treating everything as strings.

schema_customer = StructType([
    StructField("id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("password", StringType(), True),
    StructField("street", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zip_code", StringType(), True),
])
schema_customer

StructType([StructField('id', IntegerType(), True), StructField('first_name', StringType(), True), StructField('last_name', StringType(), True), StructField('email', StringType(), True), StructField('password', StringType(), True), StructField('street', StringType(), True), StructField('city', StringType(), True), StructField('state', StringType(), True), StructField('zip_code', StringType(), True)])

In [69]:
print(DUMMY_DESTINATION_FILE_CUSTOMER_FULL_PATH)
df_customer = spark.read.csv(DUMMY_DESTINATION_FILE_CUSTOMER_FULL_PATH, header=True, schema=schema_customer)
df_customer.show()

hdfs://dists-hdfs-namenode:8020/assiation-rule-mining/customer.csv
+----+----------+---------+--------------------+--------+----------------+-----------+----------+--------+
|  id|first_name|last_name|               email|password|          street|       city|     state|zip_code|
+----+----------+---------+--------------------+--------+----------------+-----------+----------+--------+
|   1|      John|      Doe|john.doe@example.com|  pwd123|     123 Main St|Pico Rivera|California|   90660|
|   2|      Jane|    Smith|jane.smith@exampl...|  pwd456|     456 Oak Ave|        Los|      NULL|    NULL|
|NULL|California|    90001|                NULL|    NULL|            NULL|       NULL|      NULL|    NULL|
|   3|       Bob|  Johnson|   bob.j@example.com|  pwd789|     789 Pine St|     Dallas|     Texas|   75201|
|   4|     Alice| Williams| alice.w@example.com|  pwd321|    321 Cedar Rd|    Seattle|Washington|   98101|
|   5|       Tom|    Brown|   tom.b@example.com|  pwd654|  654 Birch Blvd|Pic

In [74]:
df_fullname = df_customer.withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))
df_fullname = df_fullname.drop("first_name", "last_name", "email", "password")
df_fullname.show()

+----+----------------+-----------+----------+--------+----------------+
|  id|          street|       city|     state|zip_code|       full_name|
+----+----------------+-----------+----------+--------+----------------+
|   1|     123 Main St|Pico Rivera|California|   90660|        John Doe|
|   2|     456 Oak Ave|        Los|      NULL|    NULL|      Jane Smith|
|NULL|            NULL|       NULL|      NULL|    NULL|California 90001|
|   3|     789 Pine St|     Dallas|     Texas|   75201|     Bob Johnson|
|   4|    321 Cedar Rd|    Seattle|Washington|   98101|  Alice Williams|
|   5|  654 Birch Blvd|Pico Rivera|California|   90660|       Tom Brown|
|   6|      987 Elm St|   New York|  New York|   10001|     Lisa Taylor|
|   7|    111 Maple Dr|     Austin|     Texas|   73301|       David Lee|
|   8|   222 Walnut Ln|      Miami|   Florida|   33101|    Susan Miller|
|   9|333 Chestnut Ave|    Chicago|  Illinois|   60601|    Chris Wilson|
|  10|      444 Ash St|Pico Rivera|California|   90

In [77]:
df_alifonia_pico_rivera = df_fullname.filter(
    (col("state") == "California") & (col("city") == "Pico Rivera")
)
df_alifonia_pico_rivera.show()

+---+--------------+-----------+----------+--------+-----------+
| id|        street|       city|     state|zip_code|  full_name|
+---+--------------+-----------+----------+--------+-----------+
|  1|   123 Main St|Pico Rivera|California|   90660|   John Doe|
|  5|654 Birch Blvd|Pico Rivera|California|   90660|  Tom Brown|
| 10|    444 Ash St|Pico Rivera|California|   90660|Nancy Moore|
+---+--------------+-----------+----------+--------+-----------+



In [79]:
df_state_counts = df_fullname.groupBy("state").count()
df_state_counts.show()

+----------+-----+
|     state|count|
+----------+-----+
|     Texas|    2|
|      NULL|    2|
|Washington|    1|
|  Illinois|    1|
|   Florida|    1|
|California|    3|
|  New York|    1|
+----------+-----+



In [109]:
CUSTOMER_STATE_COUNTS_HDFS_OUTPUT_PATH = (
    HDFS_NAMENODE 
    + DUMMY_DESTITAION_DIR
    + "/reports/customer-state-count/"
    + datetime.now().strftime("%Y-%m-%d")
)
CUSTOMER_STATE_COUNTS_HDFS_OUTPUT_PATH

'hdfs://dists-hdfs-namenode:8020/assiation-rule-mining/reports/customer-state-count/2026-04-25'

In [90]:
df_state_counts.write.format("parquet").mode("overwrite").save(CUSTOMER_STATE_COUNTS_HDFS_OUTPUT_PATH)

In [98]:
# Tinih diem sinh vien
# Analalysts

df_student_score = spark.read.csv(DUMMY_DESTINATION_FILE_STUDENT_SCORE_FULL_PATH, header=True, inferSchema=True)
df_student_score.show(5)

+----+---+---+----+---+---+----+---+---+----+---+
|MaSV|LTC|TRR|CTDL| GT|HĐH|CSDL|MMT|WCB|XSTK|AI1|
+----+---+---+----+---+---+----+---+---+----+---+
|SV01|9.0|8.5| 7.5|8.0|7.0| 6.5|9.0|8.5| 8.0|9.0|
|SV02|6.0|5.5| 6.5|5.0|7.5| 6.0|6.0|5.5| 7.0|6.0|
|SV03|8.5|9.0| 8.5|8.5|9.0| 7.5|9.5|9.0| 8.5|9.0|
|SV04|7.0|7.5| 6.0|8.0|6.5| 8.5|7.0|8.5| 6.0|8.0|
|SV05|8.0|8.5| 8.0|7.0|8.5| 7.0|8.5|8.5| 9.0|8.0|
+----+---+---+----+---+---+----+---+---+----+---+
only showing top 5 rows



In [108]:
#   Tạo cột TRANS bao gồm các môn học có điểm >= 8.0
columns = df_student_score.columns
subject_columns = columns[1:]

conditions = [when(col(subject) >= 8.0, lit(subject)) for subject in subject_columns]

df_score_greater_than_eight = df_student_score.withColumn("TRANS", concat_ws(",", *conditions))
df_score_greater_than_eight = df_score_greater_than_eight.withColumn("TRANS", split(col("TRANS"), ","))
df_score_greater_than_eight.show(5)

[Stage 4:>                                                          (0 + 1) / 1]

+----+---+---+----+---+---+----+---+---+----+---+--------------------+
|MaSV|LTC|TRR|CTDL| GT|HĐH|CSDL|MMT|WCB|XSTK|AI1|               TRANS|
+----+---+---+----+---+---+----+---+---+----+---+--------------------+
|SV01|9.0|8.5| 7.5|8.0|7.0| 6.5|9.0|8.5| 8.0|9.0|[LTC, TRR, GT, MM...|
|SV02|6.0|5.5| 6.5|5.0|7.5| 6.0|6.0|5.5| 7.0|6.0|                  []|
|SV03|8.5|9.0| 8.5|8.5|9.0| 7.5|9.5|9.0| 8.5|9.0|[LTC, TRR, CTDL, ...|
|SV04|7.0|7.5| 6.0|8.0|6.5| 8.5|7.0|8.5| 6.0|8.0|[GT, CSDL, WCB, AI1]|
|SV05|8.0|8.5| 8.0|7.0|8.5| 7.0|8.5|8.5| 9.0|8.0|[LTC, TRR, CTDL, ...|
+----+---+---+----+---+---+----+---+---+----+---+--------------------+
only showing top 5 rows



In [111]:
STUDENT_SCORE_GREATER_THAN_EIGHT_HDFS_OUTPUT_PATH = (
    HDFS_NAMENODE 
    + DUMMY_DESTITAION_DIR
    + "/reports/student-score-greater-than-eight/"
    + datetime.now().strftime("%Y-%m-%d")
)
df_score_greater_than_eight.write.format("parquet").mode("overwrite").save(STUDENT_SCORE_GREATER_THAN_EIGHT_HDFS_OUTPUT_PATH)

In [119]:
# Accociation rules:
fpGrowth = FPGrowth(itemsCol="TRANS", minSupport=5/df_score_greater_than_eight.count(), minConfidence=0.8)
model = fpGrowth.fit(df_score_greater_than_eight)

In [121]:
rules = model.associationRules
rules.orderBy("confidence", ascending=False).show(truncate=False)

+---------------------------+----------+----------+------------------+-------+
|antecedent                 |consequent|confidence|lift              |support|
+---------------------------+----------+----------+------------------+-------+
|[CTDL, WCB]                |[XSTK]    |1.0       |1.6666666666666667|0.5    |
|[CTDL, LTC, MMT, XSTK, WCB]|[HĐH]     |1.0       |2.0               |0.5    |
|[CTDL, WCB]                |[LTC]     |1.0       |1.6666666666666667|0.5    |
|[MMT, GT, WCB]             |[XSTK]    |1.0       |1.6666666666666667|0.5    |
|[CTDL, WCB]                |[MMT]     |1.0       |1.6666666666666667|0.5    |
|[HĐH, LTC]                 |[CTDL]    |1.0       |2.0               |0.5    |
|[CTDL, WCB]                |[AI1]     |1.0       |1.4285714285714286|0.5    |
|[HĐH, LTC]                 |[MMT]     |1.0       |1.6666666666666667|0.5    |
|[CTDL, WCB]                |[HĐH]     |1.0       |2.0               |0.5    |
|[HĐH, LTC]                 |[AI1]     |1.0       |1